# Processing the data (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [2]:
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install --upgrade pip
# !pip install torch
# !pip install transformers[torch]
# !pip install scikit-learn
# !pip install matplotlib
# !pip install wandb

In [3]:
from datasets import load_dataset
import csv

# 1. Define the column names according to the LIAR dataset description
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

# 2. Load local files
raw_datasets = load_dataset(
    "csv", 
    data_files={
        "train": "/home/onyxia/work/Stat_App/Data/train.tsv", 
        "validation": "/home/onyxia/work/Stat_App/Data/valid.tsv", 
        "test": "/home/onyxia/work/Stat_App/Data/test.tsv"
    }, 
    delimiter="\t", 
    column_names=col_names,
    quoting=csv.QUOTE_NONE
)

# 3. Create a mapping for the labels (Text -> Integer)
label_mapping = {
    'pants-fire': 0, 
    'false': 1, 
    'barely-true': 2, 
    'half-true': 3, 
    'mostly-true': 4, 
    'true': 5
}

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

raw_datasets = raw_datasets.map(map_labels)

print(raw_datasets)

/home/onyxia/work/venv_sa/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 10269
    })
    validation: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1283
    })
})


In [4]:
from transformers import AutoTokenizer, DataCollatorWithPadding

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["statement"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# cols_to_keep = ["input_ids", "attention_mask", "label"]
# tokenized_datasets = tokenized_datasets.remove_columns(
#     [c for c in tokenized_datasets["train"].column_names if c not in cols_to_keep]
# )

# Formatage pour PyTorch
#tokenized_datasets.set_format("torch")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# FINE TUNING

In [6]:
import wandb

wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true
wandb: Paste an API key from your profile and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/onyxia/.netrc
wandb: Currently logged in as: hadrien-vacher (hadrien-vacher-ensae) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    # On prend l'index de la plus haute probabilité (argmax) pour trouver la classe prédite (0 à 5)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [14]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

# 1. Initialiser le projet WandB
# Changez le nom si vous voulez regrouper vos expériences différemment
wandb.init(project="liar-fake-news-detection", name="bert-base-finetuning-v1")

# 2. Recharger le modèle (pour être sûr qu'il est vierge avant l'entraînement)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=6)

# 3. Arguments d'entraînement adaptés pour le Plotting
training_args = TrainingArguments(
    output_dir="liar-bert-finetuned-wandb",
    
    # --- Configuration du Logging (Pour avoir de belles courbes) ---
    report_to="wandb",          # Envoie les logs vers Weights & Biases
    logging_strategy="steps",   # Log par étape (et pas juste par époque)
    logging_steps=10,           # Enregistre un point tous les 10 batchs (très précis)
    
    # --- Stratégie d'évaluation ---
    eval_strategy="steps",      # Évalue aussi par étapes pour voir la courbe de validation
    eval_steps=50,              # Vérifie la validation tous les 50 batchs
    save_steps=100,             # Sauvegarde un checkpoint tous les 100 steps
    
    # --- Paramètres standards ---
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# 4. Initialiser le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,             
    compute_metrics=compute_metrics, 
)

# 5. Lancer l'entraînement
trainer.train()

# 6. Fermer proprement l'expérience WandB à la fin
wandb.finish()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_22476/189841080.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy
50,1.772700,1.754034,0.207165
100,1.736500,1.728268,0.235202
150,1.707300,1.719047,0.221963
200,1.704600,1.705329,0.257009
250,1.689200,1.703532,0.248442
300,1.706500,1.694523,0.251558
350,1.628600,1.693401,0.245327
400,1.646700,1.684159,0.270249
450,1.644000,1.689991,0.252336
500,1.609600,1.680314,0.266355


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/accuracy,▁▄▃▆▅▆▅█▆▇▇█
eval/loss,█▆▅▃▃▂▂▁▂▁▁▁
eval/runtime,▁▂▃▅▅▇▆▆▇▇█▇
eval/samples_per_second,█▇▆▄▄▂▃▃▂▂▁▁
eval/steps_per_second,█▇▆▄▄▂▃▃▂▂▁▁
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,▄▁▂▁▂▄▇▃▄▃▄▄▂▅▂▃▄▁▄█▃▂▄▇▄▆▅▃▇▅▄▆▄▄▆▅▅▅▄▅
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁
train/loss,█▇▆▇▇▅▅▅▆▄▅▄▄▄▄▄▄▅▅▅▅▄▃▂▂▃▂▂▂▁▂▁▃▂▂▂▂▂▁▁
eval/accuracy,0.27181
